# 顧客分析｜顧客分級與輪廓洞察

商業問題：  
顧客等級與價值分析  
分析方法：  
- 依會員等級分析營收占比與平均消費
- 以 `Gini 係數`評估顧客消費價值集中程度

In [32]:
import pandas as pd
import numpy as np
df=pd.read_csv("customers.csv")
tier_summary=df.groupby("Customer_Tier")["Total_Spent"].agg(["sum","count"]).reset_index()
tier_summary["Revenue_Share"]=tier_summary["sum"]/tier_summary["sum"].sum()*100
tier_summary["Avg_Spent"]=tier_summary["sum"]/tier_summary["count"]
def calculate_gini(series):
    x=np.sort(series.values)
    n=len(x)
    if n==0 or np.sum(x)==0:
        return 0.0
    index=np.arange(1,n+1)
    return (2*np.sum(index*x)-(n+1)*np.sum(x))/(n*np.sum(x))
overall_gini=calculate_gini(df["Total_Spent"])
gini_by_tier=df.groupby("Customer_Tier")["Total_Spent"].apply(calculate_gini).reset_index(name="Gini_Coefficient")
final_result=pd.merge(tier_summary,gini_by_tier,on="Customer_Tier")
final_result=final_result.rename(columns={"sum":"Total_Spent","count":"Customer_Count"})
print("全體 Gini 係數:", overall_gini)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)
print(final_result)

全體 Gini 係數: 0.4705115253863612
  Customer_Tier   Total_Spent  Customer_Count  Revenue_Share      Avg_Spent  Gini_Coefficient
0          Gold  2.324346e+08            6692       4.902063   34733.199432          0.142927
1      Platinum  4.455365e+09           27349      93.963992  162907.785311          0.322794
2        Silver  5.376676e+07            5959       1.133945    9022.782907          0.377753


分析結果：  
全體 Gini 係數 0.47 ，顯示顧客消費金額存在一定程度的集中現象。 Platinum 貢獻近 94% 營收，但內部 Gini（0.32） 也偏高； Silver 雖收入佔比僅 1.1% ，內部不均程度反而最高（ 0.38 ），代表低等級中仍有少數消費特別突出的客戶， Gold 內部分布最平均。公司營收主要依賴 Platinum ， Gold 會員彼此消費金額差異不大，但 Silver 會員雖收入占比僅 1.1% ，但內部消費分布的不均程度最高，顯示其中存在少數消費明顯較高的會員，值得進一步拆解其消費特徵。

商業問題：  
RFM 顧客價值分級  
分析方法：  
- 依會員編號計算 `RFM` 三項指標  
- RFM 五分位評分與顧客分級  
- 依 RFM 評分進行顧客分群

In [1]:
import pandas as pd
sales=pd.read_csv("sales.csv")
sales=sales[sales["Order_Status"]=="Delivered"].copy()
sales["Order_Date"]=pd.to_datetime(sales["Order_Date"],format="%d/%m/%Y")
ref_date=sales["Order_Date"].max()+pd.Timedelta(days=1)
rfm=sales.groupby("Customer_ID").agg(
Recency=("Order_Date",lambda x:(ref_date-x.max()).days),
Frequency=("Order_ID","count"),
Monetary=("Total_Amount","sum")).reset_index()
rfm["R_Score"]=pd.qcut(rfm["Recency"],5,labels=[5,4,3,2,1]).astype(int)
rfm["F_Score"]=pd.qcut(rfm["Frequency"].rank(method="first"),5,labels=[1,2,3,4,5]).astype(int)
rfm["M_Score"]=pd.qcut(rfm["Monetary"],5,labels=[1,2,3,4,5]).astype(int)
def segment_by_cell(row):
    r, f, m = row["R_Score"],row["F_Score"], row["M_Score"]
    if r>=4 and f>=4 and m>=4:
        return "主力客群"
    elif r<=2 and f>=4 and m>=4:
        return "高價值但久未消費"
    elif r>=4 and f==1:
        return "近期低頻消費"
    else:
        return "一般顧客"
rfm["Customer_Segment"]=rfm.apply(segment_by_cell,axis=1)
rfm.to_csv("rfm_result.csv", index=False, encoding="utf-8-sig")
print(rfm["Customer_Segment"].value_counts().sort_values(ascending=False))

Customer_Segment
一般顧客        30568
主力客群         5206
高價值但久未消費     2313
近期低頻消費       1639
Name: count, dtype: int64


分析結果：  
透過 RFM 模型將 39,726 位客戶分群，此分群以 Recency、Frequency、Monetary 三項指標為基礎，透過五分位數（Quintile）評分後進行顧客分級，兼顧客戶的活躍度與消費貢獻。結果顯示，一般顧客占多數（ 30,568 位）；主力客群有 5,206 位，屬於近期、高頻且高消費的客戶；高價值但久未消費的客戶有 2,313 位，過去消費頻率與金額較高，但近期活躍度較低；近期低頻顧客有 1,639 位，屬於近期有消費但購買頻率較低的客群，可進一步觀察其後續回購行為。整體而言，此 RFM 分群可區分不同活躍度與消費價值的顧客群，作為後續顧客經營、回購與留存分析的基礎。

商業問題：  
不同客群的商品類別消費差異  
分析方法：  
- 依年齡層、商品類別彙整營收
- 計算各客群與商品類別的營收占比
- 比較年齡層與商品類別的營收結構

In [24]:
import pandas as pd
sales=pd.read_csv("sales.csv")
products=pd.read_csv("products.csv")
customers=pd.read_csv("customers.csv")
df_1=sales.merge(customers[["Customer_ID","Age_Group"]],on="Customer_ID",how="inner")
df_1=df_1.merge(products[["Product_ID","Category"]],on="Product_ID",how="inner")
pivot_rev=pd.pivot_table(
    df_1,
    values="Total_Amount",
    index="Age_Group",
    columns="Category",
    aggfunc="sum",
    fill_value=0
)
pct_total=(pivot_rev/df_1["Total_Amount"].sum()*100).round(2)
pct_row=pivot_rev.div(pivot_rev.sum(axis=1), axis=0).mul(100).round(2)
pct_col=pivot_rev.div(pivot_rev.sum(axis=0), axis=1).mul(100).round(2)
print("佔全公司總營收百分比 (%)")
print(pct_total)
print("佔各年齡層內部百分比 (列加總 100%)")
print(pct_row)
print("佔各商品類別內部百分比 (欄加總 100%)")
print(pct_col)

佔全公司總營收百分比 (%)
Category   Beauty  Books  Electronics  Fashion  Grocery  Home  Sports
Age_Group                                                            
18-25        0.00   0.31        22.46     2.32     0.00  0.00    0.00
26-35        0.00   0.00        31.91     3.31     0.00  0.00    8.02
36-45        0.00   0.00        18.24     0.00     0.35  6.02    0.00
46-55        0.39   0.00         0.00     0.00     0.21  3.64    0.00
56-65        0.19   0.00         0.00     0.00     0.11  1.84    0.00
65+          0.06   0.00         0.00     0.00     0.03  0.58    0.00
佔各年齡層內部百分比 (列加總 100%)
Category   Beauty  Books  Electronics  Fashion  Grocery   Home  Sports
Age_Group                                                             
18-25        0.00   1.25        89.51     9.23     0.00   0.00    0.00
26-35        0.00   0.00        73.79     7.65     0.00   0.00   18.55
36-45        0.00   0.00        74.11     0.00     1.42  24.46    0.00
46-55        9.21   0.00         0.00     0.00  

分析結果：  
建立年齡層 × 商品類別的營收樞紐表，再從三個角度觀察營收結構：佔全公司總營收比例、佔各年齡層內部比例，以及佔各商品類別內部比例。結果顯示， 18–45 歲客群的商品類別營收主要集中於電子產品，其中電子產品在各年齡層的內部營收占比均較高； 46 歲以上客群則以居家用品為主要消費品類，占該年齡層營收約 86% 。從商品類別的客群結構來看，書籍類別的營收完全集中於 18–25 歲客群，運動類別則主要由 26–35 歲客群貢獻；美妝類別則以 46–55 歲客群的貢獻比例最高（60.57%）。整體而言，不同商品類別的主要營收來源存在明顯的年齡層差異，可作為後續客群與商品類別經營、商品推薦及行銷策略規劃的參考。

商業問題：  
地區顧客分布與消費力分析  
分析方法： 
- 依 `State` 計算會員數與平均消費  
- 以`中位數`區分會員密度與平均消費高低
- 依`會員密度`與`平均消費`將地區分為四類
- 統計各類地區的邦數

In [3]:
import pandas as pd
customers=pd.read_csv("customers.csv")
sales=pd.read_csv("sales.csv")
merged=customers.merge(sales,on="Customer_ID",how="left")
state_df=(merged.groupby("State_x").agg(
        customer_count=("Customer_ID", "nunique"),
        avg_spending=("Total_Amount", "mean"),).round(2).reset_index()
    .rename(columns={'State_x': 'State'}))
density_median=state_df["customer_count"].median()
spending_median=state_df["avg_spending"].median()
def classify_state(row):
    density="高規模" if row["customer_count"]>=density_median else "低規模"
    spending="高消費" if row["avg_spending"]>=spending_median else "低消費"
    return f"{density}×{spending}"
state_df["分類四分區"]=state_df.apply(classify_state, axis=1)
result_df=state_df.sort_values(by=["customer_count","avg_spending"], ascending=False)
print("人口規模中位數：",density_median)
print("平均消費中位數：",spending_median)
print(result_df)
summary_df=(state_df.groupby("分類四分區").agg(包含邦數=("State","count")))
print("四分區各別邦數計算：")
print(summary_df)

人口規模中位數： 4038.0
平均消費中位數： 23759.415
         State  customer_count  avg_spending    分類四分區
8           UP            5218      23546.66  高規模×低消費
2      Haryana            5124      23553.37  高規模×低消費
6    Rajasthan            5057      23914.81  高規模×高消費
5       Punjab            4083      23745.81  高規模×低消費
4  Maharashtra            4050      23778.58  高規模×高消費
1      Gujarat            4026      23804.29  低規模×高消費
7   Tamil Nadu            3170      23773.02  低規模×高消費
9  West Bengal            3140      23461.18  低規模×低消費
0        Delhi            3116      24301.85  低規模×高消費
3    Karnataka            3016      23397.85  低規模×低消費
四分區各別邦數計算：
         包含邦數
分類四分區        
低規模×低消費     2
低規模×高消費     3
高規模×低消費     3
高規模×高消費     2


分析結果：  
以各邦會員規模與平均訂單金額的中位數（ 4,038 人、 23,759 元）進行四分區，結果顯示各邦分布較為分散，其中高規模低消費與低規模高消費各有 3 邦。整體而言，會員規模與平均訂單金額呈現交錯分布，會員規模較大的地區不一定具有較高的平均訂單金額，顯示兩項指標並非單純的同向關係。